# 2. Feature Extraction
Runs MediaPipe Holistic over every video in `wlasl_quarter.csv` and saves one `.npy` keypoint array per video.

**Note:** pinned to `mediapipe==0.10.21` — newer versions (0.10.30+) removed the legacy `mp.solutions` API this notebook uses. If you already have a newer mediapipe installed, run the install cell then **restart the runtime** before continuing (Colab caches the old import).

In [ ]:
!pip install -q mediapipe==0.10.21 opencv-python tqdm pandas numpy

## Restart runtime now if you just installed/changed mediapipe
Runtime -> Restart session, then re-run from here.

In [ ]:
import os

import cv2
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
import mediapipe as mp

mp_holistic = mp.solutions.holistic
print("mediapipe version:", mp.__version__)

## Keypoint extraction
Pose (33*4) + face (468*3) + left hand (21*3) + right hand (21*3) = 1662 dims per frame.

In [ ]:
def _landmarks_to_array(landmarks, n_points, n_dims):
    if landmarks is None:
        return np.zeros(n_points * n_dims, dtype=np.float32)
    if n_dims == 4:
        vals = [[p.x, p.y, p.z, p.visibility] for p in landmarks.landmark]
    else:
        vals = [[p.x, p.y, p.z] for p in landmarks.landmark]
    return np.array(vals, dtype=np.float32).flatten()


def extract_keypoints(results) -> np.ndarray:
    pose = _landmarks_to_array(results.pose_landmarks, 33, 4)
    face = _landmarks_to_array(results.face_landmarks, 468, 3)
    lh = _landmarks_to_array(results.left_hand_landmarks, 21, 3)
    rh = _landmarks_to_array(results.right_hand_landmarks, 21, 3)
    return np.concatenate([pose, face, lh, rh])


def extract_features_from_video(video_path: str, max_frames: int = None) -> np.ndarray:
    cap = cv2.VideoCapture(video_path)
    frame_features = []

    with mp_holistic.Holistic(min_detection_confidence=0.5,
                               min_tracking_confidence=0.5) as holistic:
        while cap.isOpened():
            ret, frame = cap.read()
            if not ret:
                break

            image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            image.flags.writeable = False
            results = holistic.process(image)

            frame_features.append(extract_keypoints(results))

            if max_frames and len(frame_features) >= max_frames:
                break

    cap.release()

    if len(frame_features) == 0:
        return np.zeros((1, 1662), dtype=np.float32)

    return np.stack(frame_features).astype(np.float32)

## Quick sanity check on one video

In [ ]:
CSV_PATH = "wlasl_quarter.csv"
OUT_DIR = "extracted_features"

df = pd.read_csv(CSV_PATH)
os.makedirs(OUT_DIR, exist_ok=True)

sample_row = df.iloc[0]
sample_feats = extract_features_from_video(sample_row["local_path"])
print("Sample video:", sample_row["video_id"], "gloss:", sample_row["gloss"])
print("Extracted feature shape:", sample_feats.shape)  # (num_frames, 1662)

## Extract features for every video in the subset
Safe to re-run — already-extracted `.npy` files are skipped.

In [ ]:
skipped, done, failed = 0, 0, 0

for _, row in tqdm(df.iterrows(), total=len(df)):
    video_id = str(row["video_id"])
    out_path = os.path.join(OUT_DIR, f"{video_id}.npy")

    if os.path.exists(out_path):
        skipped += 1
        continue

    try:
        features = extract_features_from_video(row["local_path"])
        np.save(out_path, features)
        done += 1
    except Exception as e:
        print(f"Failed on {video_id}: {e}")
        failed += 1

print(f"Done: {done} extracted, {skipped} already existed, {failed} failed")